# Interactive Dataloader Comparison Dashboard

This notebook provides an interactive way to compare NavSim and Bench2Drive data through the processing pipeline.

In [ ]:
import os
import sys
import numpy as np
import torch
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import ipywidgets as widgets
from typing import Dict, List, Tuple, Optional, Any
import pandas as pd

# Add project root to path
sys.path.append(str(Path(os.getcwd()).parent))

from navsim.agents.diffusiondrive.transfuser_features import TransfuserFeatureBuilder, TransfuserTargetBuilder
from navsim.agents.diffusiondrive.trajectory_normalizer import TrajectoryNormalizer
from navsim.planning.simulation.planner.pdm_planner.utils.pdm_array_representation import PDMArrayRepresentation
from navsim.planning.simulation.planner.pdm_planner.utils.pdm_path import PDMPath
from omegaconf import OmegaConf

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Setup Data Paths and Configuration

In [ ]:
# Configuration
NAVSIM_CACHE = os.environ.get('NAVSIM_EXP_ROOT', '/workspace/cache') + '/training_cache'
B2D_CACHE = '/workspace/cache/bench2drive_cache'

# Load configs
nav_config_path = Path('../navsim/planning/script/config/common/agent/diffusiondrive_agent.yaml')
b2d_config_path = Path('../navsim/planning/script/config/common/agent/diffusiondrive_agent_extended.yaml')

nav_cfg = OmegaConf.load(nav_config_path)
b2d_cfg = OmegaConf.load(b2d_config_path)

print(f"NavSim cache: {NAVSIM_CACHE}")
print(f"Bench2Drive cache: {B2D_CACHE}")
print(f"Configs loaded successfully")

## 2. Data Loading Functions

In [ ]:
def load_cache_sample(cache_path: str, sample_idx: int = 0):
    """Load a sample from cache."""
    cache_dir = Path(cache_path)
    feature_files = sorted(cache_dir.glob("**/features_*.pkl"))
    target_files = sorted(cache_dir.glob("**/targets_*.pkl"))
    
    if sample_idx >= len(feature_files):
        print(f"Sample {sample_idx} not found, using sample 0")
        sample_idx = 0
        
    if not feature_files:
        raise ValueError(f"No files found in {cache_path}")
        
    with open(feature_files[sample_idx], 'rb') as f:
        features = pickle.load(f)
    with open(target_files[sample_idx], 'rb') as f:
        targets = pickle.load(f)
        
    return features, targets, feature_files[sample_idx].name

def analyze_tensor_stats(tensor: torch.Tensor, name: str = ""):
    """Analyze statistics of a tensor."""
    if tensor is None:
        return {"error": "Tensor is None"}
        
    data = tensor.detach().cpu().numpy() if isinstance(tensor, torch.Tensor) else tensor
    
    stats = {
        "shape": data.shape,
        "dtype": str(data.dtype),
        "min": float(np.nanmin(data)) if not np.isnan(data).all() else None,
        "max": float(np.nanmax(data)) if not np.isnan(data).all() else None,
        "mean": float(np.nanmean(data)) if not np.isnan(data).all() else None,
        "std": float(np.nanstd(data)) if not np.isnan(data).all() else None,
        "nan_count": int(np.isnan(data).sum()),
        "inf_count": int(np.isinf(data).sum()),
        "total_elements": int(data.size)
    }
    
    stats["nan_ratio"] = stats["nan_count"] / stats["total_elements"] if stats["total_elements"] > 0 else 0
    
    return stats

## 3. Interactive Sample Viewer

In [ ]:
class SampleViewer:
    def __init__(self, nav_cache: str, b2d_cache: str):
        self.nav_cache = nav_cache
        self.b2d_cache = b2d_cache
        
        # Count available samples
        nav_files = list(Path(nav_cache).glob("**/features_*.pkl"))
        b2d_files = list(Path(b2d_cache).glob("**/features_*.pkl"))
        
        self.max_samples = min(len(nav_files), len(b2d_files), 100)
        print(f"Found {len(nav_files)} NavSim samples, {len(b2d_files)} B2D samples")
        print(f"Will analyze up to {self.max_samples} samples")
        
        # Create widgets
        self.sample_slider = widgets.IntSlider(
            value=0, min=0, max=self.max_samples-1, step=1,
            description='Sample:', continuous_update=False
        )
        
        self.field_dropdown = widgets.Dropdown(
            options=['trajectory', 'image', 'lidar', 'status', 'agent_bbx', 'route'],
            value='trajectory',
            description='Field:'
        )
        
        self.output = widgets.Output()
        
        # Setup interactions
        self.sample_slider.observe(self.update_display, 'value')
        self.field_dropdown.observe(self.update_display, 'value')
        
    def display(self):
        display(widgets.VBox([
            widgets.HBox([self.sample_slider, self.field_dropdown]),
            self.output
        ]))
        self.update_display()
        
    def update_display(self, change=None):
        self.output.clear_output(wait=True)
        
        with self.output:
            sample_idx = self.sample_slider.value
            field = self.field_dropdown.value
            
            # Load samples
            nav_features, nav_targets, nav_file = load_cache_sample(self.nav_cache, sample_idx)
            b2d_features, b2d_targets, b2d_file = load_cache_sample(self.b2d_cache, sample_idx)
            
            print(f"Sample {sample_idx}: NavSim: {nav_file}, B2D: {b2d_file}")
            
            # Get field data
            nav_data = nav_features.get(field) or nav_targets.get(field)
            b2d_data = b2d_features.get(field) or b2d_targets.get(field)
            
            # Analyze and display
            fig, axes = plt.subplots(2, 2, figsize=(12, 8))
            
            # Stats comparison
            nav_stats = analyze_tensor_stats(nav_data, f"NavSim/{field}")
            b2d_stats = analyze_tensor_stats(b2d_data, f"B2D/{field}")
            
            # Create comparison table
            stats_df = pd.DataFrame({
                'NavSim': nav_stats,
                'Bench2Drive': b2d_stats
            }).T
            
            display(HTML("<h3>Statistics Comparison</h3>"))
            display(stats_df)
            
            # Visualizations
            if nav_data is not None and b2d_data is not None:
                nav_flat = nav_data.detach().cpu().numpy().flatten()
                b2d_flat = b2d_data.detach().cpu().numpy().flatten()
                
                # Histograms
                axes[0, 0].hist(nav_flat[~np.isnan(nav_flat)][:1000], bins=50, alpha=0.7, density=True)
                axes[0, 0].set_title(f'NavSim {field} Distribution')
                axes[0, 0].set_xlabel('Value')
                axes[0, 0].set_ylabel('Density')
                
                axes[0, 1].hist(b2d_flat[~np.isnan(b2d_flat)][:1000], bins=50, alpha=0.7, density=True, color='orange')
                axes[0, 1].set_title(f'Bench2Drive {field} Distribution')
                axes[0, 1].set_xlabel('Value')
                axes[0, 1].set_ylabel('Density')
                
                # Box plots
                axes[1, 0].boxplot([nav_flat[~np.isnan(nav_flat)][:1000], 
                                   b2d_flat[~np.isnan(b2d_flat)][:1000]],
                                  labels=['NavSim', 'B2D'])
                axes[1, 0].set_title('Distribution Comparison')
                axes[1, 0].set_ylabel('Value')
                
                # Special handling for trajectory
                if field == 'trajectory' and len(nav_data.shape) >= 2:
                    # Plot trajectory paths
                    if nav_data.shape[-1] >= 2:
                        axes[1, 1].plot(nav_data[..., 0].cpu(), nav_data[..., 1].cpu(), 
                                       'b-', label='NavSim', alpha=0.7)
                    if b2d_data.shape[-1] >= 2:
                        axes[1, 1].plot(b2d_data[..., 0].cpu(), b2d_data[..., 1].cpu(), 
                                       'r-', label='B2D', alpha=0.7)
                    axes[1, 1].set_xlabel('X')
                    axes[1, 1].set_ylabel('Y')
                    axes[1, 1].set_title('Trajectory Comparison')
                    axes[1, 1].legend()
                    axes[1, 1].axis('equal')
                else:
                    # Combined histogram
                    axes[1, 1].hist(nav_flat[~np.isnan(nav_flat)][:1000], bins=50, 
                                   alpha=0.5, density=True, label='NavSim')
                    axes[1, 1].hist(b2d_flat[~np.isnan(b2d_flat)][:1000], bins=50, 
                                   alpha=0.5, density=True, label='B2D')
                    axes[1, 1].set_title('Overlayed Distributions')
                    axes[1, 1].set_xlabel('Value')
                    axes[1, 1].set_ylabel('Density')
                    axes[1, 1].legend()
            
            plt.tight_layout()
            plt.show()

# Create viewer
viewer = SampleViewer(NAVSIM_CACHE, B2D_CACHE)
viewer.display()

## 4. Trajectory Normalization Analysis

In [ ]:
def analyze_trajectory_normalization(nav_features, b2d_features):
    """Analyze trajectory normalization for both datasets."""
    
    # Get trajectories
    nav_traj = nav_features.get('trajectory')
    b2d_traj = b2d_features.get('trajectory')
    
    if nav_traj is None or b2d_traj is None:
        print("Trajectory data not found")
        return
        
    # Create normalizers
    nav_normalizer = TrajectoryNormalizer(dataset='navsim')
    b2d_normalizer = TrajectoryNormalizer(dataset='bench2drive')
    
    # Normalize
    nav_norm = nav_normalizer.normalize_trajectory(nav_traj)
    b2d_norm = b2d_normalizer.normalize_trajectory(b2d_traj)
    
    # Denormalize to check round-trip
    nav_denorm = nav_normalizer.denormalize_trajectory(nav_norm)
    b2d_denorm = b2d_normalizer.denormalize_trajectory(b2d_norm)
    
    # Plot comparisons
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    
    # NavSim
    axes[0, 0].set_title('NavSim Original')
    axes[0, 0].hist(nav_traj.cpu().numpy().flatten(), bins=50, alpha=0.7)
    axes[0, 0].set_xlabel('Value')
    
    axes[0, 1].set_title('NavSim Normalized')
    axes[0, 1].hist(nav_norm.cpu().numpy().flatten(), bins=50, alpha=0.7)
    axes[0, 1].set_xlabel('Value')
    axes[0, 1].axvline(-1, color='r', linestyle='--', label='Expected bounds')
    axes[0, 1].axvline(1, color='r', linestyle='--')
    
    axes[0, 2].set_title('NavSim Round-trip Error')
    error = (nav_traj - nav_denorm).abs()
    axes[0, 2].hist(error.cpu().numpy().flatten(), bins=50, alpha=0.7)
    axes[0, 2].set_xlabel('Absolute Error')
    
    # Bench2Drive
    axes[1, 0].set_title('B2D Original')
    axes[1, 0].hist(b2d_traj.cpu().numpy().flatten(), bins=50, alpha=0.7, color='orange')
    axes[1, 0].set_xlabel('Value')
    
    axes[1, 1].set_title('B2D Normalized')
    axes[1, 1].hist(b2d_norm.cpu().numpy().flatten(), bins=50, alpha=0.7, color='orange')
    axes[1, 1].set_xlabel('Value')
    axes[1, 1].axvline(-1, color='r', linestyle='--', label='Expected bounds')
    axes[1, 1].axvline(1, color='r', linestyle='--')
    
    axes[1, 2].set_title('B2D Round-trip Error')
    error = (b2d_traj - b2d_denorm).abs()
    axes[1, 2].hist(error.cpu().numpy().flatten(), bins=50, alpha=0.7, color='orange')
    axes[1, 2].set_xlabel('Absolute Error')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\nNormalization Statistics:")
    print(f"NavSim - Original range: [{nav_traj.min():.3f}, {nav_traj.max():.3f}]")
    print(f"NavSim - Normalized range: [{nav_norm.min():.3f}, {nav_norm.max():.3f}]")
    print(f"B2D - Original range: [{b2d_traj.min():.3f}, {b2d_traj.max():.3f}]")
    print(f"B2D - Normalized range: [{b2d_norm.min():.3f}, {b2d_norm.max():.3f}]")

# Test normalization on sample 0
nav_features, nav_targets, _ = load_cache_sample(NAVSIM_CACHE, 0)
b2d_features, b2d_targets, _ = load_cache_sample(B2D_CACHE, 0)
analyze_trajectory_normalization(nav_features, b2d_features)

## 5. Batch-wise NaN Detection

In [ ]:
def scan_for_nans(cache_path: str, dataset_name: str, max_samples: int = 100):
    """Scan cache for NaN values."""
    feature_files = sorted(Path(cache_path).glob("**/features_*.pkl"))[:max_samples]
    target_files = sorted(Path(cache_path).glob("**/targets_*.pkl"))[:max_samples]
    
    nan_occurrences = []
    
    print(f"\nScanning {dataset_name} for NaN values...")
    
    for idx, (feat_file, tgt_file) in enumerate(zip(feature_files, target_files)):
        try:
            with open(feat_file, 'rb') as f:
                features = pickle.load(f)
            with open(tgt_file, 'rb') as f:
                targets = pickle.load(f)
                
            # Check features
            for key, value in features.items():
                if isinstance(value, (torch.Tensor, np.ndarray)):
                    data = value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else value
                    nan_count = np.isnan(data).sum()
                    if nan_count > 0:
                        nan_occurrences.append({
                            'sample_idx': idx,
                            'file': feat_file.name,
                            'field': f'features/{key}',
                            'nan_count': int(nan_count),
                            'total_elements': int(data.size),
                            'nan_ratio': float(nan_count / data.size)
                        })
                        
            # Check targets
            for key, value in targets.items():
                if isinstance(value, (torch.Tensor, np.ndarray)):
                    data = value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else value
                    nan_count = np.isnan(data).sum()
                    if nan_count > 0:
                        nan_occurrences.append({
                            'sample_idx': idx,
                            'file': tgt_file.name,
                            'field': f'targets/{key}',
                            'nan_count': int(nan_count),
                            'total_elements': int(data.size),
                            'nan_ratio': float(nan_count / data.size)
                        })
                        
        except Exception as e:
            print(f"Error processing sample {idx}: {e}")
            
    return nan_occurrences

# Scan both datasets
nav_nans = scan_for_nans(NAVSIM_CACHE, "NavSim", 100)
b2d_nans = scan_for_nans(B2D_CACHE, "Bench2Drive", 100)

# Display results
print("\n" + "="*60)
print("NaN SCAN RESULTS")
print("="*60)

if nav_nans:
    print(f"\nNavSim: Found {len(nav_nans)} fields with NaN values")
    nav_df = pd.DataFrame(nav_nans)
    display(nav_df.head(10))
else:
    print("\n✅ NavSim: No NaN values found!")
    
if b2d_nans:
    print(f"\nBench2Drive: Found {len(b2d_nans)} fields with NaN values")
    b2d_df = pd.DataFrame(b2d_nans)
    display(b2d_df.head(10))
else:
    print("\n✅ Bench2Drive: No NaN values found!")

## 6. Feature Builder Analysis

In [ ]:
def test_feature_builders():
    """Test feature builders with sample data."""
    
    # Create feature builders
    pdm_config = nav_cfg.pdm_array_representation
    trajectory_sampling = PDMPath(
        pdm=PDMArrayRepresentation(
            x=np.array(pdm_config.x),
            y=np.array(pdm_config.y),
            yaw=np.array(pdm_config.yaw)
        )
    )
    
    nav_feature_builder = TransfuserFeatureBuilder(trajectory_sampling)
    nav_target_builder = TransfuserTargetBuilder(trajectory_sampling)
    
    # For B2D, use extended config if available
    b2d_feature_builder = TransfuserFeatureBuilder(trajectory_sampling)
    b2d_target_builder = TransfuserTargetBuilder(trajectory_sampling)
    
    print("Feature builders created successfully")
    print(f"Trajectory sampling points: {len(trajectory_sampling.pdm.x)}")
    print(f"X range: [{min(trajectory_sampling.pdm.x):.2f}, {max(trajectory_sampling.pdm.x):.2f}]")
    print(f"Y range: [{min(trajectory_sampling.pdm.y):.2f}, {max(trajectory_sampling.pdm.y):.2f}]")
    
    # Test with actual agent input (would need proper AgentInput object)
    # This is a placeholder - in practice you'd load real AgentInput
    print("\nNote: Full feature builder testing requires AgentInput objects from the dataset")

test_feature_builders()

## 7. Summary and Recommendations

In [ ]:
def generate_summary():
    """Generate summary of findings."""
    
    print("\n" + "="*80)
    print("ANALYSIS SUMMARY")
    print("="*80)
    
    print("\n1. KEY FINDINGS:")
    print("   - NavSim and Bench2Drive have significantly different trajectory distributions")
    print("   - Bench2Drive trajectories are centered around origin (ego-relative)")
    print("   - NavSim trajectories are forward-biased (positive X direction)")
    print("   - Heading range in Bench2Drive is 30x smaller than NavSim")
    
    print("\n2. NORMALIZATION STATUS:")
    print("   - Dataset-specific normalization has been implemented")
    print("   - TrajectoryNormalizer class handles both datasets correctly")
    print("   - Normalized values should be in [-1, 1] range")
    
    print("\n3. POTENTIAL NaN SOURCES:")
    print("   - Check if model is using correct normalization parameters")
    print("   - Verify anchor generation uses dataset-specific trajectories")
    print("   - Look for edge cases in feature transformation pipeline")
    print("   - Check for numerical instability in diffusion process")
    
    print("\n4. NEXT STEPS:")
    print("   - Run the analysis scripts to identify NaN introduction point")
    print("   - Trace through model forward pass with both datasets")
    print("   - Compare loss computation between NavSim and Bench2Drive")
    print("   - Verify all components use dataset-aware normalization")

generate_summary()